In [1]:
from transformers import AutoTokenizer
import sentencepiece as spm

## Data loading and splitting

In [2]:
from speakleash import Speakleash
import os

base_dir = os.getcwd()
replicate_to = os.path.join(base_dir, "datasets")

sl = Speakleash(replicate_to)

In [3]:
# for d in sorted(sl.datasets, key=lambda x: x.characters):
#     size_mb = round(d.characters / 1024 / 1024)
#     print(f"{d.name}: {size_mb} MB, {d.documents} documents")

In [4]:
max_chars = 60 * 1024 * 1024
written = 0

with (
    open("plwiki_train.txt", "w", encoding="utf-8") as f_train,
    open("plwiki_eval.txt", "w", encoding="utf-8") as f_eval,
):
    for i, doc in enumerate(sl.get("plwiki").data):
        f = f_eval if i % 10 == 0 else f_train
        f.write(doc.strip() + "\n\n")
        written += len(doc)
        if written >= max_chars:
            break

print("Characters written:", written)

Characters written: 62927191


In [5]:
# Documents are separated by a blank line; train and eval come from disjoint documents
def read_docs(path):
    with open(path, encoding="utf-8") as f:
        return [d for d in f.read().split("\n\n") if d.strip()]


train_docs = read_docs("plwiki_train.txt")
eval_docs = read_docs("plwiki_eval.txt")

print(len(train_docs), len(eval_docs))

10543 1172


## Tokenizers

Pretrained GPT2 BPE tokenizer

In [6]:
gpt_bpe_tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [10]:
text = "Ledwo nad wierzchem ręką znak pożegnania odfrunie, nowe piasków sklepienie wiatr wzdyma w ciemnej łunie."

# Text -> list of token strings
tokens = gpt_bpe_tokenizer.tokenize(text)
print(tokens)

# Text -> list of token IDs
ids = gpt_bpe_tokenizer.encode(text)
print(ids)

# Token IDs -> text
print(gpt_bpe_tokenizer.decode(ids))

# Size of the vocabulary
print(gpt_bpe_tokenizer.vocab_size)

['Led', 'wo', 'Ġn', 'ad', 'Ġw', 'ier', 'z', 'chem', 'Ġr', 'Ä', 'Ļ', 'k', 'Ä', 'ħ', 'Ġz', 'n', 'ak', 'Ġpo', 'Å', '¼', 'eg', 'n', 'ania', 'Ġo', 'df', 'run', 'ie', ',', 'Ġnow', 'e', 'Ġpi', 'ask', 'Ã³', 'w', 'Ġsk', 'le', 'p', 'ien', 'ie', 'Ġw', 'iat', 'r', 'Ġw', 'z', 'dy', 'ma', 'Ġw', 'Ġc', 'iem', 'ne', 'j', 'ĠÅ', 'Ĥ', 'un', 'ie', '.']
[42416, 21638, 299, 324, 266, 959, 89, 15245, 374, 128, 247, 74, 128, 227, 1976, 77, 461, 745, 129, 120, 1533, 77, 5411, 267, 7568, 5143, 494, 11, 783, 68, 31028, 2093, 10205, 86, 1341, 293, 79, 2013, 494, 266, 5375, 81, 266, 89, 9892, 2611, 266, 269, 26597, 710, 73, 25370, 224, 403, 494, 13]
Ledwo nad wierzchem ręką znak pożegnania odfrunie, nowe piasków sklepienie wiatr wzdyma w ciemnej łunie.
50257


Whitespace-based tokenizer (custom implementation)

In [9]:
import os

from custom_tokenizer import WhitespaceTokenizer

WS_PATH = "ws_tokenizer.json"

if os.path.exists(WS_PATH):
    # Reuse the vocabulary saved by a previous run
    ws_tokenizer = WhitespaceTokenizer.load(WS_PATH)
else:
    # Same N as the GPT-2 BPE tokenizer (includes the <UNK> token)
    ws_tokenizer = WhitespaceTokenizer(vocab_size=gpt_bpe_tokenizer.vocab_size)
    ws_tokenizer.train(train_docs)
    ws_tokenizer.save(WS_PATH)

text = "Ledwo nad wierzchem ręką znak pożegnania odfrunie, nowe piasków sklepienie wiatr wzdyma w ciemnej łunie."

# Text -> list of token strings (punctuation is a separate token)
print(ws_tokenizer.tokenize(text))

# Text -> list of token IDs (out-of-vocabulary tokens map to <UNK>, id 0)
ids = ws_tokenizer.encode(text)
print(ids)

# Token IDs -> text
print(ws_tokenizer.decode(ids))

# Size of the vocabulary
print(len(ws_tokenizer.token_to_id))

['ledwo', 'nad', 'wierzchem', 'ręką', 'znak', 'pożegnania', 'odfrunie', ',', 'nowe', 'piasków', 'sklepienie', 'wiatr', 'wzdyma', 'w', 'ciemnej', 'łunie', '.']
[38343, 76, 0, 11526, 1473, 43943, 0, 2, 430, 23019, 22857, 3777, 0, 3, 17902, 0, 1]
ledwo nad <UNK> ręką znak pożegnania <UNK>, nowe piasków sklepienie wiatr <UNK> w ciemnej <UNK>.
50257


SentencePiece tokenizer

In [13]:
# Train on the same data as the other tokenizers (plwiki_train.txt). Documents are
# split into lines: SentencePiece skips sentences longer than max_sentence_length (4192 bytes)
def train_lines():
    for article in train_docs:
        for line in article.split("\n"):
            if line.strip():
                yield line.strip()


# Training writes sp.model and sp.vocab, so skip it when they already exist
if not os.path.exists("sp.model"):
    spm.SentencePieceTrainer.train(
        sentence_iterator=train_lines(),
        model_prefix="sp",  # creates sp.model and sp.vocab
        vocab_size=gpt_bpe_tokenizer.vocab_size,  # same N as the other tokenizers
        model_type="unigram",  # or "bpe"
        input_sentence_size=300_000,  # sample of lines, keeps memory and time reasonable
        shuffle_input_sentence=True,
    )

In [12]:
# Load the trained model
sp = spm.SentencePieceProcessor(model_file="sp.model")

text = "Ledwo nad wierzchem ręką znak pożegnania odfrunie, nowe piasków sklepienie wiatr wzdyma w ciemnej łunie."
print(sp.encode(text, out_type=str))
ids = sp.encode(text, out_type=int)  # token ids
print(ids)
print(sp.decode(ids))
print(sp.get_piece_size())

['▁Le', 'dwo', '▁nad', '▁wierzch', 'em', '▁ręką', '▁znak', '▁pożegna', 'nia', '▁od', 'fr', 'u', 'nie', ',', '▁nowe', '▁piasków', '▁sklepienie', '▁wiatr', '▁w', 'zd', 'ym', 'a', '▁w', '▁ciemnej', '▁łu', 'nie', '.']
[1010, 14091, 112, 15633, 36, 18957, 1975, 18567, 111, 28, 2754, 22, 76, 4, 716, 35943, 28508, 3637, 5, 3680, 134, 9, 5, 32373, 9756, 76, 3]
Ledwo nad wierzchem ręką znak pożegnania odfrunie, nowe piasków sklepienie wiatr wzdyma w ciemnej łunie.
50257


## Model training (Transformers)

In [15]:
from train_lm import TrainConfig, encode_corpus

config = TrainConfig(vocab_size=gpt_bpe_tokenizer.vocab_size)


def lines(docs):
    return [line for article in docs for line in article.split("\n") if line.strip()]


# Every tokenizer encodes exactly the same text
train_texts = lines(train_docs)[:200_000]
val_texts = lines(eval_docs)
val_chars = sum(len(t) for t in val_texts)

tokenizers = {
    "bpe": gpt_bpe_tokenizer.encode,
    "whitespace": ws_tokenizer.encode,
    "sentencepiece": sp.encode,
}

# Encode once and cache the ids on disk
encoded = {}
for name, encode in tokenizers.items():
    tr = encode_corpus(encode, train_texts, f"ids_{name}_train.npy")
    va = encode_corpus(encode, val_texts, f"ids_{name}_val.npy")
    encoded[name] = (tr, va)
    print(f"{name}: {len(tr):,} train tokens, {len(va):,} validation tokens")

bpe: 28,488,267 train tokens, 3,125,075 validation tokens
whitespace: 9,488,374 train tokens, 1,038,403 validation tokens
sentencepiece: 11,512,600 train tokens, 1,282,224 validation tokens


In [ ]:
# Run a short test first (e.g. config.max_steps = 20) to measure the time per step

# for name, (tr, va) in encoded.items():
#     train(name, tr, va, val_chars, config)

/home/pawlowiczf/Natural-language-processing/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


bpe: 30.0M parameters on cpu


: 